In [1]:
import pandas as pd

### Metadata mapping transposed
Generated at step 06.02_Cancer_and_treatment_cleaning

In [3]:
old_metadata_T = pd.read_csv('./Previous/metadata_mapping_transposed.csv')
new_metadata_T = pd.read_csv('./New/metadata_mapping_transposed.csv')
print(f'Len PREVIOUS metadata: {len(old_metadata_T)}')
print(f'Len NEW metadata: {len(new_metadata_T)}')

Len PREVIOUS metadata: 12157
Len NEW metadata: 2621


In [ ]:
metadata_T = pd.concat([old_metadata_T, new_metadata_T], axis=0, ignore_index=True)
l_metadata_before_drop = len(metadata_T)
metadata_T = metadata_T.drop_duplicates()
print(f'Len metadata updated: {len(metadata_T)}')
print(f'Duplicated rows: {l_metadata_before_drop - len(metadata_T)}')

entities_multiple_categories = (
    metadata_T.groupby('Entity')['Category']
      .nunique()
      .loc[lambda x: x > 1]
)

print(f'Entities with more than one Category: {len(entities_multiple_categories)}')
print(entities_multiple_categories)

metadata_T.to_csv('./Updated/metadata_mapping_transposed.csv', sep=',', index=False)
print('Saved file to ./Updated/metadata_mapping_transposed.csv')

metadata = metadata_T.set_index('Entity').T
metadata.to_csv('./Updated/metadata_mapping.csv', sep=',', index=False)
print('Saved file to ./Updated/metadata_mapping.csv')

metadata_T

Len metadata updated: 13892
Duplicated rows: 886
Entities with more than one Category: 0
Series([], Name: Category, dtype: int64)
Saved file to ./Updated/metadata_mapping_transposed.csv
Saved file to ./Updated/metadata_mapping.csv


Entity,PaperTitle,Study_design,Study_weight,Abstract,PaperId,"2,4-pyrimidinediamine",3-Dimensional Conformal Radiation Therapy,4'-(9-acridinylamino)methanesulfon-m-anisidide,5-Fluorouracil/Salicylic Acid Topical Solution,7-Ethyl-10-Hydroxycamptothecin,...,5649_Treatment,625_Treatment,9121_Treatment,6281_Treatment,9236_Treatment,17370_Treatment,6534_Treatment,astroblastoma,histiocytic sarcoma,uterine leiomyosarcoma
Category,Initial df,Study,Study,Initial df,Initial df,Treatment,Treatment,Treatment,Treatment,Treatment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cancer,Cancer,Cancer


### Final variant treatment consensus
Generated at step 06.02.5_LLM-based_coassociation

In [2]:
old_variant_treatment_consensus = pd.read_csv('./Previous/final_variant_treatment_consensus.csv')
new_variant_treatment_consensus = pd.read_csv('./New/final_variant_treatment_consensus.csv')
print(f'Len PREVIOUS variant treatment consensus: {len(old_variant_treatment_consensus)}')
print(f'Len NEW variant treatment consensus: {len(new_variant_treatment_consensus)}')

Len PREVIOUS variant treatment consensus: 15624
Len NEW variant treatment consensus: 619


In [3]:
variant_treatment_consensus = pd.concat([old_variant_treatment_consensus,
                                        new_variant_treatment_consensus],
                                        axis=0, ignore_index=True)
l_variant_treatment_consensus_before_drop = len(variant_treatment_consensus)
variant_treatment_consensus = variant_treatment_consensus.drop_duplicates()
print(f'Len variant_treatment_consensus updated: {len(variant_treatment_consensus)}')
print(f'Duplicated rows: {l_variant_treatment_consensus_before_drop - len(variant_treatment_consensus)}')

Len variant_treatment_consensus updated: 16088
Duplicated rows: 155


In [4]:
pairs_multiple_pred = (
    variant_treatment_consensus.groupby('Variant_Treatment_Pair')['Resolved_Prediction']
      .nunique()
      .loc[lambda x: x > 1]
)

print(f'Entities with more than one Category: {len(pairs_multiple_pred)}')

comparison = old_variant_treatment_consensus.merge(
    new_variant_treatment_consensus,
    on='Variant_Treatment_Pair',
    how='outer',
    suffixes=('_old', '_new')
)

changed = comparison[
    (comparison['Resolved_Prediction_old'].notna()) &
    (comparison['Resolved_Prediction_new'].notna()) &
    (comparison['Resolved_Prediction_old'] != comparison['Resolved_Prediction_new'])
]

print(changed.value_counts(
    ['Resolved_Prediction_old', 'Resolved_Prediction_new']
)
)

changed

Entities with more than one Category: 127
Resolved_Prediction_old  Resolved_Prediction_new
Sensitive                Unrelated                  20
Resistant                Sensitive                  18
                         Unknown                    15
                         Unrelated                  15
Sensitive                Resistant                  14
                         Unknown                    12
Unknown                  Unrelated                   6
Unrelated                Sensitive                   5
Unknown                  Sensitive                   4
                         Resistant                   4
Resistant                No consensus                3
No consensus             Sensitive                   2
Sensitive                No consensus                2
Diagnostic               Unrelated                   2
Unrelated                Unknown                     2
Diagnostic               Unknown                     1
No consensus             Resi

,Variant_Treatment_Pair,Resolved_Prediction_old,Resolved_Prediction_new
99,156157insalu_brca2 + adjuvant chemotherapy,Unknown,Sensitive
100,156157insalu_brca2 + chemotherapy,Unknown,Sensitive
878,a289v_egfr + temozolomide,Diagnostic,Unknown
2798,e17k_akt1 + paclitaxel,Sensitive,Resistant
3066,e542k_pik3ca + tamoxifen,Resistant,Unknown
...,...,...,...
14886,v600k_braf + trametinib,Sensitive,Unknown
14920,v600r_braf + trametinib,Sensitive,Unknown
15762,y537s_esr1 + capecitabine,Sensitive,Unrelated
15774,y537s_esr1 + fulvestrant,Resistant,No consensus


### Solve consensus between predictions

In [5]:
import regex as re

# Extract LLM resposne
def extract_variant_treatment_prediction(response):
    """Extracts (Variant, Treatment, Prediction) tuples from a raw LLM response."""
    results = []
    if pd.isna(response) or not isinstance(response, str):
        return results
    for line in response.split("\n"):
        match = re.match(r"(.+?)\s*\+\s*(.+?)\s*:\s*(\w+)", line.strip())
        if match:
            variant = match.group(1).strip()
            treatment = match.group(2).strip()
            prediction = match.group(3).strip()
            results.append((variant, treatment, prediction))
    return results

# Define the extraction function
def extract_variant_treatment_prediction(response):
    results = []
    if pd.isna(response) or not isinstance(response, str):
        return results
    for line in response.split("\n"):
        match = re.match(r"(.+?)\s*\+\s*(.+?)\s*:\s*(\w+)", line.strip())
        if match:
            variant = match.group(1).strip()
            treatment = match.group(2).strip()
            prediction = match.group(3).strip()
            results.append((variant, treatment, prediction))
    return results

# Clean variant strings
def clean_variant(variant):
    if pd.isna(variant):
        return ""
    variant = variant.strip().lower()
    variant = re.sub(r"[^\w\s]", "", variant)
    variant = re.sub(r"\s+", "", variant)
    variant = variant.replace("__", "_")
    return variant

def process_variant_coassociation_LLM_df(variant_coassociation_LLM_df):
    variant_coassociation_LLM_df.columns = variant_coassociation_LLM_df.columns.str.strip()
    variant_coassociation_LLM_df["Parsed_Triples"] = variant_coassociation_LLM_df["LLM_Response"].apply(extract_variant_treatment_prediction)
    flat_records = []
    for idx, row in variant_coassociation_LLM_df.iterrows():
        paper_id = row.get("PaperId", None)
        for variant, treatment, prediction in row["Parsed_Triples"]:
            flat_records.append({
                "PaperId": paper_id,
                "Variant": variant,
                "Treatment": treatment,
                "Prediction": prediction
            })

    df_llm_extracted = pd.DataFrame(flat_records)
    print(df_llm_extracted.head())
    print(f"\nTotal variant-treatment-prediction entries extracted: {len(df_llm_extracted):,}")

    variant_coassociation_LLM_df["Parsed_Triples"] = variant_coassociation_LLM_df["LLM_Response"].apply(extract_variant_treatment_prediction)
    flat_records = []
    for idx, row in variant_coassociation_LLM_df.iterrows():
        paper_id = row.get("PaperId", None)
        parsed_triples = row.get("Parsed_Triples", [])
        for variant, treatment, prediction in parsed_triples:
            flat_records.append({
                "PaperId": paper_id,
                "Variant": variant,
                "Treatment": treatment,
                "Prediction": prediction
            })

    df_llm_extracted = pd.DataFrame(flat_records)
    print(" Preview of extracted variant-treatment-prediction entries:")
    print(df_llm_extracted.head())
    print(f"\nTotal variant-treatment-prediction entries extracted: {len(df_llm_extracted):,}")

    df_llm_extracted["Variant_Clean"] = df_llm_extracted["Variant"].apply(clean_variant)

    # Create Variant_Treatment_Pair column
    df_llm_extracted["Variant_Treatment_Pair"] = (
        df_llm_extracted["Variant_Clean"].str.strip() + " + " +
        df_llm_extracted["Treatment"].str.strip().str.lower()
    )

    return df_llm_extracted

In [6]:
old_variant_coassociation_LLM_df = pd.read_csv("./Previous/LLM_variant_screening_llama33-70b_prompt1.csv")
old_LLM_extracted = process_variant_coassociation_LLM_df(old_variant_coassociation_LLM_df)
old_LLM_extracted

     PaperId     Variant    Treatment Prediction
0  100806778   v600_BRAF  Binimetinib  Resistant
1  100806778   v600_BRAF  Palbociclib  Unrelated
2  100806778   v600_BRAF   Ribociclib  Unrelated
3  100806778   v600_BRAF   Trametinib  Resistant
4  100806778  v600e_BRAF  Binimetinib  Resistant

Total variant-treatment-prediction entries extracted: 37,963
 Preview of extracted variant-treatment-prediction entries:
     PaperId     Variant    Treatment Prediction
0  100806778   v600_BRAF  Binimetinib  Resistant
1  100806778   v600_BRAF  Palbociclib  Unrelated
2  100806778   v600_BRAF   Ribociclib  Unrelated
3  100806778   v600_BRAF   Trametinib  Resistant
4  100806778  v600e_BRAF  Binimetinib  Resistant

Total variant-treatment-prediction entries extracted: 37,963


,PaperId,Variant,Treatment,Prediction,Variant_Clean,Variant_Treatment_Pair
0,100806778,v600_BRAF,Binimetinib,Resistant,v600_braf,v600_braf + binimetinib
1,100806778,v600_BRAF,Palbociclib,Unrelated,v600_braf,v600_braf + palbociclib
2,100806778,v600_BRAF,Ribociclib,Unrelated,v600_braf,v600_braf + ribociclib
3,100806778,v600_BRAF,Trametinib,Resistant,v600_braf,v600_braf + trametinib
4,100806778,v600e_BRAF,Binimetinib,Resistant,v600e_braf,v600e_braf + binimetinib
...,...,...,...,...,...,...
37958,970466766,l858r_EGFR,Erlotinib,Sensitive,l858r_egfr,l858r_egfr + erlotinib
37959,988899048,t393c_GNAS,Chemotherapy,Unrelated,t393c_gnas,t393c_gnas + chemotherapy
37960,988899048,t393c_GNAS,Erlotinib,Diagnostic,t393c_gnas,t393c_gnas + erlotinib
37961,988899048,t393c_GNAS,Gefitinib,Diagnostic,t393c_gnas,t393c_gnas + gefitinib


In [7]:
new_variant_coassociation_LLM_df = pd.read_csv("./New/LLM_variant_screening_llama33-70b_prompt1.csv")
new_LLM_extracted = process_variant_coassociation_LLM_df(new_variant_coassociation_LLM_df)
new_LLM_extracted

      PaperId     Variant              Treatment Prediction
0  4406282429  v600e_BRAF          Immunotherapy  Sensitive
1  4406282429  v637e_BRAF          Immunotherapy    Unknown
2  4406834107   g12d_KRAS          Immunotherapy    Unknown
3  4406834107   g12d_KRAS  Ionizing Radiotherapy  Sensitive
4  4406834107   g12d_KRAS      Radiation Therapy  Sensitive

Total variant-treatment-prediction entries extracted: 812
 Preview of extracted variant-treatment-prediction entries:
      PaperId     Variant              Treatment Prediction
0  4406282429  v600e_BRAF          Immunotherapy  Sensitive
1  4406282429  v637e_BRAF          Immunotherapy    Unknown
2  4406834107   g12d_KRAS          Immunotherapy    Unknown
3  4406834107   g12d_KRAS  Ionizing Radiotherapy  Sensitive
4  4406834107   g12d_KRAS      Radiation Therapy  Sensitive

Total variant-treatment-prediction entries extracted: 812


,PaperId,Variant,Treatment,Prediction,Variant_Clean,Variant_Treatment_Pair
0,4406282429,v600e_BRAF,Immunotherapy,Sensitive,v600e_braf,v600e_braf + immunotherapy
1,4406282429,v637e_BRAF,Immunotherapy,Unknown,v637e_braf,v637e_braf + immunotherapy
2,4406834107,g12d_KRAS,Immunotherapy,Unknown,g12d_kras,g12d_kras + immunotherapy
3,4406834107,g12d_KRAS,Ionizing Radiotherapy,Sensitive,g12d_kras,g12d_kras + ionizing radiotherapy
4,4406834107,g12d_KRAS,Radiation Therapy,Sensitive,g12d_kras,g12d_kras + radiation therapy
...,...,...,...,...,...,...
807,7122542093,r172h_TP53,Selumetinib,Sensitive,r172h_tp53,r172h_tp53 + selumetinib
808,7124732292,g12c_KRAS,Avelumab,Unknown,g12c_kras,g12c_kras + avelumab
809,7124732292,g12c_KRAS,Radiation Therapy,Unknown,g12c_kras,g12c_kras + radiation therapy
810,7124732292,g12d_KRAS,Avelumab,Unknown,g12d_kras,g12d_kras + avelumab


In [8]:
df_llm_extracted = pd.concat([old_LLM_extracted, new_LLM_extracted], axis=0, ignore_index=True)

# Count predictions per pair
prediction_counts = (
    df_llm_extracted
    .groupby(["Variant_Treatment_Pair", "Prediction"])
    .size()
    .reset_index(name="Count")
)

# Total predictions per pair
total_counts = (
    df_llm_extracted
    .groupby("Variant_Treatment_Pair")
    .size()
    .reset_index(name="Total")
)

merged = prediction_counts.merge(total_counts, on="Variant_Treatment_Pair")
merged["Is_Consensus"] = merged["Count"] == merged["Total"]
merged

,Variant_Treatment_Pair,Prediction,Count,Total,Is_Consensus
0,1013dupa_eogt + radiation therapy,Unrelated,1,1,True
1,1016del_brca1 + immune checkpoint inhibitor,Unknown,1,1,True
2,1016del_brca1 + pembrolizumab,Unknown,1,1,True
3,1017dela_hla + cyclophosphamide,Unknown,1,1,True
4,1017dela_hla + doxorubicin,Unknown,1,1,True
...,...,...,...,...,...
19424,yvma_erbb2 + neratinib,Resistant,1,1,True
19425,yvma_erbb2 + tyrosine kinase inhibitor,Resistant,1,1,True
19426,znf384_ep300 + chemotherapy,Unknown,1,1,True
19427,znf384_ep300 + hematopoietic cell transplantation,Unrelated,1,1,True


In [9]:
consensus_only = merged[merged["Is_Consensus"]].copy()
print("\n=== Variant + Treatment Pairs with 100% LLM Prediction Consensus ===")
print(consensus_only.sort_values(by="Count", ascending=False).head(20))

total_pairs = merged["Variant_Treatment_Pair"].nunique()
non_consensus_pairs = merged[~merged["Is_Consensus"]]["Variant_Treatment_Pair"].nunique()
consensus_pairs = total_pairs - non_consensus_pairs

print(f"\n Total unique Variant + Treatment pairs: {total_pairs}")
print(f" Pairs WITHOUT full consensus: {non_consensus_pairs}")
print(f" Pairs WITH full consensus: {consensus_pairs}")

# Show non-consensus rows
no_consensus = merged[~merged["Is_Consensus"]].copy()
no_consensus_sorted = no_consensus.sort_values(by="Total", ascending=False)
print("\n===  Variant + Treatment Pairs WITHOUT LLM Prediction Consensus ===")
no_consensus_sorted.head(20)


=== Variant + Treatment Pairs with 100% LLM Prediction Consensus ===
                           Variant_Treatment_Pair Prediction  Count  Total  \
1690                        c481s_btk + ibrutinib  Resistant     27     27   
17334   v600_braf + dabrafenib/trametinib regimen  Sensitive     25     25   
17561  v600e_braf + cetuximab/encorafenib regimen  Sensitive     21     21   
2420                       d538g_esr1 + letrozole  Resistant     14     14   
18044                    v600k_braf + encorafenib  Sensitive     13     13   
16220                        t315i_bcr + imatinib  Resistant     12     12   
19212                      y537s_esr1 + letrozole  Resistant     12     12   
9064                 l536h_esr1 + hormone therapy  Resistant     11     11   
7412                   h1047r_pik3ca + copanlisib  Sensitive     11     11   
1883                     c797g_egfr + osimertinib  Resistant     10     10   
6381                       g465r_egfr + cetuximab  Resistant     10     

,Variant_Treatment_Pair,Prediction,Count,Total,Is_Consensus
17965,v600e_braf + vemurafenib,Sensitive,414,534,False
17964,v600e_braf + vemurafenib,Resistant,113,534,False
17963,v600e_braf + vemurafenib,Diagnostic,1,534,False
17966,v600e_braf + vemurafenib,Unknown,6,534,False
17562,v600e_braf + chemotherapy,Diagnostic,3,502,False
17563,v600e_braf + chemotherapy,Resistant,145,502,False
17564,v600e_braf + chemotherapy,Sensitive,49,502,False
17565,v600e_braf + chemotherapy,Unknown,148,502,False
17566,v600e_braf + chemotherapy,Unrelated,157,502,False
16608,t790m_egfr + osimertinib,Resistant,60,444,False


In [ ]:
# Building consensus labels for variant-treatment predictions
valid_preds = ["Sensitive", "Resistant", "Diagnostic"]
merged["Proportion"] = merged["Count"] / merged["Total"]
merged_sorted = merged.sort_values(["Variant_Treatment_Pair", "Proportion"], ascending=[True, False])
dominant_per_pair = merged_sorted.drop_duplicates("Variant_Treatment_Pair").copy()
dominant_per_pair.loc[:, "Soft_Consensus"] = (
    (dominant_per_pair["Proportion"] >= 0.60) &
    (dominant_per_pair["Total"] >= 3) &
    (dominant_per_pair["Prediction"].isin(valid_preds))
)

soft_consensus_total = dominant_per_pair["Soft_Consensus"].sum()
soft_consensus_percent = 100 * soft_consensus_total / dominant_per_pair.shape[0]
print(f"Soft consensus pairs (custom rules): {soft_consensus_total}")
print(f"Percentage of all Variant+Treatment pairs: {soft_consensus_percent:.1f}%")

hard_consensus_pairs = set(merged.loc[merged["Is_Consensus"], "Variant_Treatment_Pair"].unique())
soft_consensus_pairs = set(dominant_per_pair.loc[dominant_per_pair["Soft_Consensus"], "Variant_Treatment_Pair"].unique())
total_pairs = df_llm_extracted["Variant_Treatment_Pair"].nunique()
all_consensus_pairs = hard_consensus_pairs.union(soft_consensus_pairs)
no_consensus_pairs = set(df_llm_extracted["Variant_Treatment_Pair"].unique()) - all_consensus_pairs
print(f"\nTotal Variant + Treatment pairs: {total_pairs}")
print(f"Hard consensus pairs: {len(hard_consensus_pairs)}")
print(f"Soft-only consensus pairs: {len(soft_consensus_pairs - hard_consensus_pairs)}")
print(f"Total pairs with any consensus (before fallback): {len(all_consensus_pairs)}")
print(f"Pairs WITHOUT any consensus (before fallback): {len(no_consensus_pairs)}")

# Fallback consensus resolution

df_no_consensus = df_llm_extracted[df_llm_extracted["Variant_Treatment_Pair"].isin(no_consensus_pairs)].copy()

fallback_results = []
for pair, group in df_no_consensus.groupby("Variant_Treatment_Pair"):
    label_counts = group["Prediction"].value_counts()
    unique_labels = label_counts.index.tolist()

    valid_labels = {"Sensitive", "Resistant", "Diagnostic"}
    weak_labels = {"Unknown", "Unrelated"}

    present_valid = [label for label in unique_labels if label in valid_labels]
    present_weak = [label for label in unique_labels if label in weak_labels]

    if set(unique_labels).issubset(weak_labels):
        if label_counts.get("Unknown", 0) > label_counts.get("Unrelated", 0):
            fallback_results.append((pair, "Unknown"))
        elif label_counts.get("Unrelated", 0) > label_counts.get("Unknown", 0):
            fallback_results.append((pair, "Unrelated"))
        else:
            fallback_results.append((pair, "Unknown"))
    elif len(unique_labels) == 1:
        fallback_results.append((pair, unique_labels[0]))
    elif present_valid and present_weak:
        top_valid_label = label_counts.loc[present_valid].idxmax()
        fallback_results.append((pair, top_valid_label))
    else:
        fallback_results.append((pair, "No consensus"))

df_fallback_consensus = pd.DataFrame(fallback_results, columns=["Variant_Treatment_Pair", "Resolved_Prediction"])

# Final consensus assembly

hard_labels = merged[merged["Is_Consensus"]].copy()
hard_labels = hard_labels.sort_values(["Variant_Treatment_Pair", "Count"], ascending=[True, False])
hard_labels = hard_labels.drop_duplicates("Variant_Treatment_Pair")[["Variant_Treatment_Pair", "Prediction"]]
hard_labels = hard_labels.rename(columns={"Prediction": "Resolved_Prediction"})

soft_labels = dominant_per_pair[dominant_per_pair["Soft_Consensus"]][["Variant_Treatment_Pair", "Prediction"]].copy()
soft_labels = soft_labels.rename(columns={"Prediction": "Resolved_Prediction"})

df_final_consensus = pd.concat([
    hard_labels,
    soft_labels[~soft_labels["Variant_Treatment_Pair"].isin(hard_labels["Variant_Treatment_Pair"])],
    df_fallback_consensus[~df_fallback_consensus["Variant_Treatment_Pair"].isin(hard_labels["Variant_Treatment_Pair"]) &
                          ~df_fallback_consensus["Variant_Treatment_Pair"].isin(soft_labels["Variant_Treatment_Pair"])]
], ignore_index=True)

df_final_consensus = df_final_consensus[["Variant_Treatment_Pair", "Resolved_Prediction"]]
df_final_consensus.to_csv("./Updated/final_variant_treatment_consensus.csv", index=False)

print("Final consensus dataset shape:")
print(df_final_consensus.shape)
print("Saved to: final_variant_treatment_consensus.csv")
df_final_consensus

Soft consensus pairs (custom rules): 812
Percentage of all Variant+Treatment pairs: 5.1%

Total Variant + Treatment pairs: 15961
Hard consensus pairs: 13629
Soft-only consensus pairs: 588
Total pairs with any consensus (before fallback): 14217
Pairs WITHOUT any consensus (before fallback): 1744
Final consensus dataset shape:
(15961, 2)
Saved to: final_variant_treatment_consensus.csv


,Variant_Treatment_Pair,Resolved_Prediction
0,1013dupa_eogt + radiation therapy,Unrelated
1,1016del_brca1 + immune checkpoint inhibitor,Unknown
2,1016del_brca1 + pembrolizumab,Unknown
3,1017dela_hla + cyclophosphamide,Unknown
4,1017dela_hla + doxorubicin,Unknown
...,...,...
15956,y772a775dup_erbb2 + trastuzumab,Sensitive
15957,y772dupyvma_erbb2 + trastuzumab deruxtecan,No consensus
15958,y806n_ret + cabozantinib,Resistant
15959,y823_kit + ponatinib,Sensitive


## Network graph weighted
Generated by step 06.04_Network_analysis_with_synonyms

Need the following files:
* metadata_mapping_transposed ✅
* cleaned_df_v4
* CIVIC_cancer_synonyms
* final_variant_treatment_consensus ✅
* CIVIC_ClinVar_merged

Merge the old version of these files with the new ones, then run the 06.04 step

### cleaned_df_v4

In [2]:
old_cleaned_df_v4 = pd.read_csv('./Previous/cleaned_df_v4.csv')
new_cleaned_df_v4 = pd.read_csv('./New/cleaned_df_v4.csv')

print(f'Shape PREVIOUS cleaned_df_v4: {old_cleaned_df_v4.shape}')
print(f'Shape NEW cleaned_df_v4: {new_cleaned_df_v4.shape}')

# Values are all 0 or 1
# old_cleaned_df_v4[list(all_cols)].stack().value_counts()
# new_cleaned_df_v4[list(all_cols)].stack().value_counts()

Shape PREVIOUS cleaned_df_v4: (7253, 12156)
Shape NEW cleaned_df_v4: (170, 2621)


In [3]:
keys = ['PaperTitle', 'Study_design', 'Study_weight', 'Abstract', 'PaperId']

cleaned_df_v4 = (
    pd.concat([
        old_cleaned_df_v4,
        new_cleaned_df_v4
    ])
    .groupby(keys, dropna=False, as_index=False)
    .max()
    .fillna(0)
)

print(f'Shape after merge and process: {cleaned_df_v4.shape}')

cleaned_df_v4.to_csv('./Updated/cleaned_df_v4.csv', index=False)

/var/folders/57/yk3w62md44z9m6ndy61mkkz80000gn/T/ipykernel_28285/2408949719.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .max()
/var/folders/57/yk3w62md44z9m6ndy61mkkz80000gn/T/ipykernel_28285/2408949719.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .max()
/var/folders/57/yk3w62md44z9m6ndy61mkkz80000gn/T/ipykernel_28285/2408949719.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc

Shape after merge and process: (7423, 13890)


### CIVIC_cancer_synonyms

In [2]:
old_CIVIC_cancer_synonyms = pd.read_csv('./Previous/CIVIC_cancer_synonyms.csv')
new_CIVIC_cancer_synonyms = pd.read_csv('./New/CIVIC_cancer_synonyms.csv')

print(f'Lenght PREVIOUS CIVIC_cancer_synonyms: {len(old_CIVIC_cancer_synonyms)}')
print(f'Lenght NEW CIVIC_cancer_synonyms: {len(new_CIVIC_cancer_synonyms)}')

CIVIC_cancer_synonyms = pd.concat([old_CIVIC_cancer_synonyms, new_CIVIC_cancer_synonyms], axis=0, ignore_index=True)
print(f'Number of duplicated rows: {CIVIC_cancer_synonyms.duplicated().sum()}')

CIVIC_cancer_synonyms = CIVIC_cancer_synonyms.drop_duplicates()
print(f'Lenght UPDATED CIVIC_cancer_synonyms (no duplicates): {len(CIVIC_cancer_synonyms)}')
CIVIC_cancer_synonyms

Lenght PREVIOUS CIVIC_cancer_synonyms: 413
Lenght NEW CIVIC_cancer_synonyms: 438
Number of duplicated rows: 408
Lenght UPDATED CIVIC_cancer_synonyms (no duplicates): 443


,id,name,doid,synonyms,diseaseUrl
0,2,Gastrointestinal Stromal Tumor,9253.0,"Gastrointestinal Stromal Tumour, Stromal Tumor...",https://www.disease-ontology.org/?id=DOID:9253
1,3,Acute Myeloid Leukemia,9119.0,"AML - Acute Myeloid Leukemia, Acute Myeloblast...",https://www.disease-ontology.org/?id=DOID:9119
2,3180,Acute Promyelocytic Leukemia With PML-RARA,81081.0,"Acute Promyelocytic Leukemia, T(15;17)(q22;q11...",https://www.disease-ontology.org/?id=DOID:0081081
3,5,Bone Marrow Cancer,4960.0,"Bone Marrow Neoplasm, Bone Marrow Tumor, Malig...",https://www.disease-ontology.org/?id=DOID:4960
4,6,Acute Leukemia,12603.0,"Stem Cell Leukaemia, Stem Cell Leukemia",https://www.disease-ontology.org/?id=DOID:12603
...,...,...,...,...,...
835,1623,Childhood Optic Nerve Glioma,6576.0,Glioma Of The Pediatric Visual Pathway,https://www.disease-ontology.org/?id=DOID:6576
836,3536,B-lymphoblastic Leukaemia/lymphoma With Other ...,NaN,NaN,NaN
837,3547,Salivary Gland Basal Cell Adenoma,NaN,NaN,NaN
839,3415,"Glioneuronal Tumor With ATRX Alteration, Kinas...",NaN,NaN,NaN


In [3]:
import pandas as pd

def merge_synonyms(series):
    vals = set()
    
    for item in series.dropna():
        parts = [s.strip() for s in str(item).split(',')]
        vals.update(p for p in parts if p)  # evita stringhe vuote
    
    if not vals:
        return pd.NA
    
    return ', '.join(sorted(vals))

CIVIC_cancer_synonyms = (
    CIVIC_cancer_synonyms
    .groupby(['name', 'doid', 'id', 'diseaseUrl'], as_index=False, dropna=False)
    .agg({'synonyms': merge_synonyms})
)

CIVIC_cancer_synonyms.to_csv('./Updated/CIVIC_cancer_synonyms.csv', index=False)
CIVIC_cancer_synonyms

,name,doid,id,diseaseUrl,synonyms
0,AML With KAT6A::CREBBP Fusion,NaN,3489,NaN,NaN
1,Acinic Cell Carcinoma,3025.0,3475,https://www.disease-ontology.org/?id=DOID:3025,NaN
2,Acral Lentiginous Melanoma,6367.0,192,https://www.disease-ontology.org/?id=DOID:6367,"Acral Lentiginous Melanoma, Malignant, Maligna..."
3,Acute Basophilic Leukemia,80795.0,3042,https://www.disease-ontology.org/?id=DOID:0080795,NaN
4,Acute Biphenotypic Leukemia,9953.0,3069,https://www.disease-ontology.org/?id=DOID:9953,Mixed Phenotype Acute Leukemia
...,...,...,...,...,...
436,Uveal Melanoma,6039.0,188,https://www.disease-ontology.org/?id=DOID:6039,Melanoma Of Uvea
437,Vagina Sarcoma,1901.0,565,https://www.disease-ontology.org/?id=DOID:1901,Sarcoma Of The Vagina
438,Villous Adenoma,50869.0,3075,https://www.disease-ontology.org/?id=DOID:0050869,NaN
439,Von Hippel-Lindau Disease,14175.0,2198,https://www.disease-ontology.org/?id=DOID:14175,NaN


In [4]:
summary = pd.DataFrame({
    'unique_values': CIVIC_cancer_synonyms.nunique(),
})

summary['duplicate_values'] = len(CIVIC_cancer_synonyms) - summary['unique_values']
summary = summary.sort_values('duplicate_values', ascending=False)

print(summary)

duplicated_names = CIVIC_cancer_synonyms[
    CIVIC_cancer_synonyms['doid'].duplicated(keep=False)
].sort_values('doid')

duplicated_names

            unique_values  duplicate_values
synonyms              257               184
doid                  387                54
diseaseUrl            388                53
id                    439                 2
name                  441                 0


,name,doid,id,diseaseUrl,synonyms
315,Obsolete PTEN Hamartoma Tumor Syndrome,80191.0,2965,https://www.disease-ontology.org/?id=DOID:0080191,NaN
334,PTEN Hamartoma Tumor Syndrome,80191.0,2965,https://www.disease-ontology.org/?id=DOID:0080191,NaN
62,B-lymphoblastic Leukemia/lymphoma With IGH::IL...,80648.0,3012,https://www.disease-ontology.org/?id=DOID:0080648,"B-ALL With IL3-IGH, B-lymphoblastic Leukemia/l..."
63,B-lymphoblastic Leukemia/lymphoma With IL3-IGH,80648.0,3012,https://www.disease-ontology.org/?id=DOID:0080648,"B-ALL With IL3-IGH, B-lymphoblastic Leukemia/l..."
0,AML With KAT6A::CREBBP Fusion,NaN,3489,NaN,NaN
7,Acute Myeloid Leukaemia With FUS::ERG Fusion,NaN,3515,NaN,NaN
10,Acute Myeloid Leukemia With CBFA2T3::GLIS2 Fusion,NaN,3474,NaN,NaN
12,Acute Myeloid Leukemia With MECOM Rearrangement,NaN,3528,NaN,NaN
14,Acute Myeloid Leukemia With MNX1::ETV6 Fusion,NaN,3483,NaN,NaN
17,Acute Myeloid Leukemia With NPM1::MLF1 Fusion,NaN,3487,NaN,NaN


### CIVIC_ClinVar_merged

In [2]:
old_CIVIC_ClinVar_merged = pd.read_csv('./Previous/CIVIC_ClinVar_merged.csv')
new_CIVIC_ClinVar_merged = pd.read_csv('./New/CIVIC_ClinVar_merged.csv')

print(f'Lenght PREVIOUS CIVIC_ClinVar_merged: {len(old_CIVIC_ClinVar_merged)}')
print(f'Lenght NEW CIVIC_ClinVar_merged: {len(new_CIVIC_ClinVar_merged)}')

CIVIC_ClinVar_merged = pd.concat([old_CIVIC_ClinVar_merged, new_CIVIC_ClinVar_merged], axis=0, ignore_index=True)
print(f'Number of duplicated rows: {CIVIC_ClinVar_merged.duplicated().sum()}')

CIVIC_ClinVar_merged = CIVIC_ClinVar_merged.drop_duplicates()
print(f'Lenght UPDATED CIVIC_cancer_synonyms (no duplicates): {len(CIVIC_ClinVar_merged)}')

CIVIC_ClinVar_merged.to_csv('./Updated/CIVIC_ClinVar_merged.csv', index=False)
CIVIC_ClinVar_merged

Lenght PREVIOUS CIVIC_ClinVar_merged: 5659
Lenght NEW CIVIC_ClinVar_merged: 6369
Number of duplicated rows: 5403
Lenght UPDATED CIVIC_cancer_synonyms (no duplicates): 6625


,Variant ID,Variant Name,Aliases,Name,Description,Score,Variants,Assertions,Gene ID,Gene Name,Gene Description,rsID,rsID_Confirmed,ClinVar_ID,HGVS_Notation,Variant_Description,Aliases_Merged
0,1,Fusion,"T(9;22)(Q34;Q11), BCR-ABL1, BCR-ABL",BCR::ABL1 Fusion,"The BCR-ABL fusion protein, commonly referred ...",353.5,Fusion,137: AID137 (B-lymphoblastic Leukemia/lymphoma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"T(9;22)(Q34;Q11), BCR-ABL1, BCR-ABL"
1,2,T315I,"THR334ILE, RS121913459",ABL1 T315I,While the efficacy of imatinib has revolutioni...,0.0,T315I,NaN,4.0,ABL1,ABL1 is most relevant to cancer in its role in...,rs121913459,rs121913459,12624.0,NM_005157.6(ABL1):c.944C>T,c.944C>T,"THR334ILE, RS121913459, NM_005157.6(ABL1):c.94..."
2,3,E255K,"E274K, RS121913448",ABL1 E255K,While the efficacy of imatinib has revolutioni...,0.0,E255K,NaN,4.0,ABL1,ABL1 is most relevant to cancer in its role in...,rs121913448,rs121913448,NaN,NaN,NaN,"E274K, RS121913448"
3,4,E17K,"GLU17LYS, RS34409589",AKT1 E17K,AKT1 E17K is a recurrent mutation that has bee...,33.5,E17K,NaN,2.0,AKT1,"AKT1, also referred to as protein kinase B, is...",rs34409589,rs34409589,1524334.0,NC_000014.9:g.104780217dup,g.104780217dup,"GLU17LYS, RS34409589, NC_000014.9:g.104780217d..."
4,5,Fusion,EML4-ALK,EML4::ALK Fusion,The EML4-ALK fusion variant 1 consisting of AL...,48.0,Fusion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EML4-ALK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12023,5856,NaN,NaN,v::TYK2 Fusion,NaN,0.0,Fusion,213: AID213 (B-lymphoblastic Leukaemia/lymphom...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12024,5857,NaN,NaN,TYK2 S431G,NaN,0.0,S431G,NaN,5970.0,TYK2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12025,5858,NaN,NaN,v::LYN Fusion,NaN,0.0,Fusion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12026,5859,NaN,NaN,TP53 V172F,NaN,0.0,V172F,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Create the Graph

In [5]:
# Set up libraries
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm

# Load the metadata and variant dataset
metadata_mapping = pd.read_csv("./Updated/metadata_mapping_transposed.csv", low_memory=False)
variant_analysis_df = pd.read_csv("./Updated/cleaned_df_v4.csv", low_memory=False)

# Cancer synonyms
CIVIC_cancer_synonyms_df = pd.read_csv("./Updated/CIVIC_cancer_synonyms.csv")
print("Cancer synonym dataset loaded successfully! Length of dataset", len(CIVIC_cancer_synonyms_df))

df_consensus = pd.read_csv("./Updated/final_variant_treatment_consensus.csv")
print("Cancer consensus dataset loaded successfully! Length of dataset", len(df_consensus))

# Get variant columns from metadata that exist in the dataset
metadata_entities = set(metadata_mapping['Entity'])
dataset_columns = set(variant_analysis_df.columns)
valid_entities = metadata_entities & dataset_columns

metadata_subset = metadata_mapping[metadata_mapping['Entity'].isin(valid_entities)]

# Separate entities by category
variants = metadata_subset[metadata_subset['Category'] == 'Variant']['Entity']
cancers = metadata_subset[metadata_subset['Category'] == 'Cancer']['Entity']
treatments = metadata_subset[metadata_subset['Category'] == 'Treatment']['Entity']

# Helper to filter existing columns with at least one '1'
def non_empty_cols(entities):
    return [col for col in entities if col in variant_analysis_df.columns and variant_analysis_df[col].sum() > 0]

variant_cols = non_empty_cols(variants)
cancer_cols = non_empty_cols(cancers)
treatment_cols = non_empty_cols(treatments)

# Reporting
print("===== Metadata Mapping Overview =====")
print(f"Total Variant entities in metadata: {variants.nunique():,}")
print(f"Total Cancer entities in metadata: {cancers.nunique():,}")
print(f"Total Treatment entities in metadata: {treatments.nunique():,}")

print("\n===== Active Columns (with at least one 1) =====")
print(f"Variant columns with at least one 1: {len(variant_cols):,}")
print(f"Cancer columns with at least one 1: {len(cancer_cols):,}")
print(f"Treatment columns with at least one 1: {len(treatment_cols):,}")

# Optionally: empty variant columns (no 1s at all)
empty_variant_cols = [col for col in variants if col in variant_analysis_df.columns and variant_analysis_df[col].sum() == 0]
print(f"\nVariant columns with NO 1s: {len(empty_variant_cols):,}")

# Filter metadata for Variant category
variant_entities = metadata_mapping[metadata_mapping['Category'] == 'Variant']['Entity']
active_variant_entities = [
    entity for entity in variant_entities
    if entity in variant_analysis_df.columns and variant_analysis_df[entity].sum() > 0
]
print(f"\nTotal active Variant entities (with at least one 1): {len(active_variant_entities)}")

# Filter metadata for Cancer category
cancer_entities = metadata_mapping[metadata_mapping['Category'] == 'Cancer']['Entity']
# Keep only active cancer entities: present in data and has at least one 1
active_cancer_entities = [
    entity for entity in cancer_entities
    if entity in variant_analysis_df.columns and variant_analysis_df[entity].sum() > 0
]
print(f"\nTotal active Cancer entities (with at least one 1): {len(active_cancer_entities)}")

# Filter metadata for Treatment category
treatment_entities = metadata_mapping[metadata_mapping['Category'] == 'Treatment']['Entity']
# Keep only active treatment entities: present in data and has at least one 1
active_treatment_entities = [
    entity for entity in treatment_entities
    if entity in variant_analysis_df.columns and variant_analysis_df[entity].sum() > 0
]
print(f"\nTotal Treatment entities defined in metadata: {treatment_entities.nunique()}")
print(f"Total active Treatment entities (with at least one 1): {len(active_treatment_entities)}\n")


Cancer synonym dataset loaded successfully! Length of dataset 441
Cancer consensus dataset loaded successfully! Length of dataset 15961
===== Metadata Mapping Overview =====
Total Variant entities in metadata: 11,676
Total Cancer entities in metadata: 162
Total Treatment entities in metadata: 623

===== Active Columns (with at least one 1) =====
Variant columns with at least one 1: 3,902
Cancer columns with at least one 1: 98
Treatment columns with at least one 1: 388

Variant columns with NO 1s: 7,774

Total active Variant entities (with at least one 1): 3902

Total active Cancer entities (with at least one 1): 98

Total Treatment entities defined in metadata: 623
Total active Treatment entities (with at least one 1): 388



In [6]:
# 1. Define allowed entities based on metadata_mapping
variant_entities = metadata_mapping[metadata_mapping['Category'] == 'Variant']['Entity'].str.strip().tolist()
cancer_entities = metadata_mapping[metadata_mapping['Category'] == 'Cancer']['Entity'].str.strip().tolist()
treatment_entities = metadata_mapping[metadata_mapping['Category'] == 'Treatment']['Entity'].str.strip().tolist()

# Combine all desired entity types
allowed_entities = set(variant_entities + cancer_entities + treatment_entities)

# 2. Define entity_columns as those in the dataset that are also in allowed_entities
entity_columns = [col for col in variant_analysis_df.columns if col in allowed_entities]

# 3. Build the graph
G = nx.Graph()

# Add nodes with category attributes
for col in entity_columns:
    if col in variant_entities:
        category = "Variant"
    elif col in cancer_entities:
        category = "Cancer"
    elif col in treatment_entities:
        category = "Treatment"
    else:
        category = "Unknown" 
    G.add_node(col, category=category)

# 4. Add edges based on co-occurrence
for idx, row in tqdm(variant_analysis_df.iterrows(), total=variant_analysis_df.shape[0], desc="Adding edges"):
    present = row[entity_columns] == 1
    active_entities = present[present].index
    for col1 in active_entities:
        for col2 in active_entities:
            if col1 != col2:
                if G.has_edge(col1, col2):
                    G[col1][col2]['weight'] += 1
                else:
                    G.add_edge(col1, col2, weight=1)

# 5. Summary and save
print("Network created!")
print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))
nx.write_gml(G, './Updated/network_graph.gml')

Adding edges: 100%|██████████| 7423/7423 [01:03<00:00, 116.25it/s]

Network created!
Nodes: 12461
Edges: 46943


In [7]:
# Verify the network analysis
G = nx.read_gml('./Updated/network_graph.gml')

# Example query: Find the neighbors of a particular variant of interest
variant_of_interest = "v600e_BRAF"
variant_neighbors = set(G.neighbors(variant_of_interest))

# Find treatments and cancers associated with the variant
treatments = []
cancers = []

for node in variant_neighbors:
    if G.nodes[node]['category'] == 'Treatment':
        treatments.append(node)
    elif G.nodes[node]['category'] == 'Cancer':
        cancers.append(node)
        
print(f"Treatments associated with variant '{variant_of_interest}':")
print(treatments)
print(f"\nCancers associated with variant '{variant_of_interest}':")
print(cancers)

# Find the top 5 most connected nodes (by degree centrality)
centrality = nx.degree_centrality(G)
sorted_centrality = sorted(centrality.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 most connected nodes (based on degree centrality):")
for node, score in sorted_centrality[:5]:
    print(f"{node}: {score:.4f}")

Treatments associated with variant 'v600e_BRAF':
['Ruxolitinib', 'Gemtuzumab Ozogamicin', 'PD173074', 'Onalespib', 'Afatinib', 'Sirolimus', 'Cetuximab/Encorafenib Regimen', 'Rucaparib', 'Mitomycin', 'MTOR Inhibitor', 'Taselisib', 'Mercaptopurine', 'Rilotumumab', 'Abemaciclib', 'Alisertib', 'Pemigatinib', 'Dacarbazine', 'Capecitabine', 'Methotrexate', 'Ivosidenib', 'Dacomitinib', 'Verteporfin', 'CDK4/6 Inhibition', 'Imatinib', 'Therapeutic Tumor Infiltrating Lymphocytes', 'Chloroquine', 'Leuprolide', 'Icotinib', 'Cytoreductive Surgery', 'Dordaviprone', 'Afimoxifene', 'Anti-EGFR Monoclonal Antibody', 'Encorafenib', 'Ramucirumab', 'Vismodegib', 'Porcupine Inhibitor WNT974', 'Lorlatinib', 'Azacitidine', 'Oxaliplatin', 'Cytarabine', 'Akt Inhibitor MK2206', 'Anti-PDL1 Therapy', 'Crizotinib', 'Hyperthermic Intraperitoneal Chemotherapy', 'Tunlametinib', 'FOLFOX Regimen', 'Trastuzumab', 'Sorafenib', 'Axitinib', 'Erlotinib', 'Sunitinib', 'Gemcitabine', 'Alvocidib', 'Ponatinib', 'Avelumab', 'Erlo

In [8]:
# 1. Define study design weights
study_design_weights = {
    'Systematic review study':       1.0,
    'Clinical study':                1.0,
    'Observational/RWE study':       0.9,
    'Case report study':             0.9,
    'In vivo/Animal study':          0.8,
    'In vitro study':                0.7,
    'In silico study':               0.6,
    'Undefined':                     0.1,
    'Other':                         0.1,
}

# 2. Identify entity columns based on metadata
variant_entities   = metadata_mapping[metadata_mapping['Category'] == 'Variant']['Entity'].str.strip().tolist()
cancer_entities    = metadata_mapping[metadata_mapping['Category'] == 'Cancer']['Entity'].str.strip().tolist()
treatment_entities = metadata_mapping[metadata_mapping['Category'] == 'Treatment']['Entity'].str.strip().tolist()

allowed_entities = set(variant_entities + cancer_entities + treatment_entities)

# 3. Extract valid binary entity columns from the dataset
non_entity_cols = ['PaperId', 'Study_design', 'Abstract', 'Study_weight', 'PaperTitle']
entity_columns = [col for col in variant_analysis_df.columns if col in allowed_entities]

# 4. Initialize graphs
G   = nx.Graph()  # unweighted
G_w = nx.Graph()  # weighted by study design

# 5. Add nodes with categories
for col in entity_columns:
    if col in variant_entities:
        cat = 'Variant'
    elif col in cancer_entities:
        cat = 'Cancer'
    elif col in treatment_entities:
        cat = 'Treatment'
    else:
        cat = 'Unknown'
    G.add_node(col, category=cat)
    G_w.add_node(col, category=cat)

# 6. Add edges based on co-occurrence and study weight
for _, row in tqdm(variant_analysis_df.iterrows(), total=len(variant_analysis_df), desc="Building graphs"):
    active_entities = row[entity_columns][row[entity_columns] == 1].index.tolist()
    design = str(row['Study_design']).strip()
    weight = study_design_weights.get(design, 0.5)

    for i, e1 in enumerate(active_entities):
        for e2 in active_entities[i+1:]:
            # Unweighted graph
            if G.has_edge(e1, e2):
                G[e1][e2]['weight'] += 1
            else:
                G.add_edge(e1, e2, weight=1)
            # Weighted graph
            if G_w.has_edge(e1, e2):
                G_w[e1][e2]['weight'] += weight
            else:
                G_w.add_edge(e1, e2, weight=weight)

# 7. Graph summaries
print("=== Unweighted graph ===")
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
deg_unw = nx.degree_centrality(G)
top_unw = sorted(deg_unw.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 nodes by degree centrality:", top_unw)

print("\n=== Weighted graph ===")
print("Nodes:", G_w.number_of_nodes(), "Edges:", G_w.number_of_edges())
deg_w = nx.degree_centrality(G_w)
top_w = sorted(deg_w.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 nodes by degree centrality:", top_w)

# 8. Compare edge weights
weights_unw = np.array([d['weight'] for _, _, d in G.edges(data=True)])
weights_w   = np.array([d['weight'] for _, _, d in G_w.edges(data=True)])

print("Edge weights (unweighted): mean=%.2f, std=%.2f" % (weights_unw.mean(), weights_unw.std()))
print("Edge weights (weighted):   mean=%.2f, std=%.2f" % (weights_w.mean(), weights_w.std()))
print("Number of edges re-weighted:", np.sum(weights_w != weights_unw))

# 9. Correlation between edge weights
corr = np.corrcoef(weights_unw, weights_w)[0, 1]
print("Pearson correlation (unweighted vs weighted): %.3f" % corr)

# 10. Degree centrality shift
delta = {n: deg_w[n] - deg_unw[n] for n in G.nodes()}

# 11. Save output
nx.write_gml(G_w, './Updated/network_graph_weighted.gml')

Building graphs: 100%|██████████| 7423/7423 [01:43<00:00, 71.80it/s]

=== Unweighted graph ===
Nodes: 12461 Edges: 46943
Top 5 nodes by degree centrality: [('Chemotherapy', 0.1267255216693419), ('breast cancer', 0.11067415730337078), ('lung cancer', 0.09285714285714286), ('colon cancer', 0.07504012841091492), ('Radiation Therapy', 0.07279293739967897)]

=== Weighted graph ===
Nodes: 12461 Edges: 46943
Top 5 nodes by degree centrality: [('Chemotherapy', 0.1267255216693419), ('breast cancer', 0.11067415730337078), ('lung cancer', 0.09285714285714286), ('colon cancer', 0.07504012841091492), ('Radiation Therapy', 0.07279293739967897)]
Edge weights (unweighted): mean=2.81, std=14.04
Edge weights (weighted):   mean=2.47, std=12.33
Number of edges re-weighted: 28730
Pearson correlation (unweighted vs weighted): 0.999
